In [30]:
import pandas as pd
import numpy as np


In [31]:
df = pd.read_csv("data/train.csv")

In [32]:
# Add label:
# 0 → A wins
# 1 → B wins
# 2 → Tie

df["label"] = (
    df["winner_model_a"] * 0 +
    df["winner_model_b"] * 1 +
    df["winner_tie"] * 2
)

df["text"] = (
    "Prompt:\n" + df["prompt"].astype(str) +
    "\n\nResponse A:\n" + df["response_a"].astype(str) +
    "\n\nResponse B:\n" + df["response_b"].astype(str)
)

In [33]:
df.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie,label,text
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0,0,"Prompt:\n[""Is it morally right to try to have ..."
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0,1,"Prompt:\n[""What is the difference between marr..."
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1,2,"Prompt:\n[""explain function calling. how would..."
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0,0,"Prompt:\n[""How can I create a test set for a v..."
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0,1,"Prompt:\n[""What is the best way to travel from..."


In [40]:
from datasets import Dataset

# 3) HF dataset
dataset = Dataset.from_pandas(df[["text", "label"]])

# 4) split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)


Map: 100%|██████████| 5748/5748 [00:01<00:00, 4760.96 examples/s]


In [41]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=1024
    )

In [42]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-large",
    num_labels=3
)

Loading weights: 100%|██████████| 390/390 [00:00<00:00, 60145.55it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

In [43]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./model",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    num_train_epochs=2,
    fp16=False,
    push_to_hub=False,
    report_to="none",
)


In [44]:


from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.722564
1000,0.000000
1500,0.000000
2000,0.000000
2500,0.000000
3000,0.000000
3500,0.000000
4000,0.000000
4500,0.000000
5000,0.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.09it/s]


In [ ]:
# Save the trained model and tokenizer
trainer.save_model("./model")
tokenizer.save_pretrained("./model")

# Evaluate and show metrics
metrics = trainer.evaluate(eval_dataset=val_dataset)
print("Validation metrics:", metrics)

# Predict to verify output shape
predictions = trainer.predict(val_dataset)
pred_labels = predictions.predictions.argmax(axis=-1)
print("Sample predictions:", pred_labels[:10])